In [2]:
import requests # This library helps us to fetch data from API
import pandas as pd # for handling and analysing data 
import numpy as np # for numerical operations 
from sklearn.model_selection import train_test_split # to split data into train and test set 
from sklearn.preprocessing import LabelEncoder # to convert catogerical data into numerical data 
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor #model for classification and regression 
from sklearn.metrics import classification_report, mean_squared_error # to evaluate the model
from datetime import datetime , timedelta # to handle date and time
from dotenv import load_dotenv
import os


In [ ]:
load_dotenv()

API_KEY = os.getenv("WEATHER_API_KEY")


In [ ]:
BASE_URL = "https://api.openweathermap.org/data/2.5/" # for making API calls to openweathermap


In [ ]:
# Fetch weather data for a specific location and date
def get_current_weather(city):
    url = f"{BASE_URL}weather?q={city}&appid={API_KEY}&units=metric"
    repsponse = requests.get(url) # making API call to fetch current weather data for the specified city
    data = repsponse.json() # converting the response to JSON format
    return {
        'city ' : data ['name'],
        'current_temp': round(data['main']['temp']),
        'feels_like': round(data['main']['feels_like']),
        'temp_min':round(data['main']['temp_min']),
        'temp_max':round(data['main']['temp_max']),
        'humidity': data['main']['humidity'],
        'description': data['weather'][0]['description'],
        'country': data['sys']['country'],
        'WindGust': data['wind']['gust'] if 'gust' in data['wind'] else None,
        'Pressure': data['main']['pressure'] if 'pressure' in data['main'] else None,
        'Wind_Gust_Speed': data['wind']['speed'] if 'speed' in data['wind'] else None
        
    }
    

In [ ]:
# Read historical data 
def read_historical_data (filename):
    df = pd.read_csv(filename)
    df= pd.dropna()
    df= pd.drop_duplicates()
    return df 

In [ ]:
# Prepare Data for training 
def prepare_data(data):
    le= LabelEncoder() # to convert categorical data into numerical data
    data['WindGustDir'] = le.fit_transform(data['WindGustDir'])
    data['RainTomorrow'] = le.fit_transform(data['RainTomorrow'])
    
    # Split data into features and target variable
    X=data.drop('RainTomorrow', axis=1) # features
    y=data['RainTomorrow'] # target variable
    return X,y,le
    
    

In [ ]:
# Train Rainfall Prediction Model
def train_rain_model(X,y):
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42) # split data into train and test set
    model = RandomForestClassifier(n_estimators=100, random_state=42) # create a random forest classifier model
    model.fit(X_train, y_train) # train the model on the training data
    y_pred = model.predict(X_test) # make predictions on the test set
    print(classification_report(y_test, y_pred)) # evaluate the model using classification report
    print("Mean Squared Error: ", mean_squared_error(y_test, y_pred)) # evaluate the model using mean squared error
    return model

In [8]:
# Prepare regression data for training
def prepare_regression_data(data, feature ):
    X,y = [],[]
    for i in range (len(data)-1):
        X.append(data[feature][i]) # features
        y.append(data[feature][i+1]) # target variable (next day's value)
    return np.array(X).reshape(-1,1), np.array(y) # reshape X to be a 2D array for regression model
    


In [9]:
# Train Regression Model for temperature prediction
def train_regression_model(X,y):
    model = RandomForestRegressor(n_estimators=100, random_state=42) # create a random forest regressor model
    model.fit(X,y) # train the model on the data
    return model

In [10]:
# predict Future Rainfall
def predict_future(model,current_value):
    predictions=[current_value]
    for i in range(5):
        next_value=model.predict(np.array([[predictions[-1]]]))# predict the next value based on the last predicted value
        predictions.append(next_value[0]) # append the predicted value to the list of predictions
    return predictions

In [ ]:
# Weather analysis Function 
def weather_view():
    city= input("Enter the city name: ") # get city name from user input
    current_weather = get_current_weather(city) # fetch current weather data for the specified city
    #Load historical data for the city
    historical_data = read_historical_data('/assets/weather.csv') # read historical
    # Prepare data for training
    X,y,le = prepare_data(historical_data) # prepare data for training
    # Train Rainfall Prediction Model
    rain_model = train_rain_model(X,y) # train the rainfall prediction model
    wind_deg= current_weather['WindGustDir'] %360 # convert wind gust direction to a value between 0 and 360
    compass_points=[
        ("N",0,11.25),
        ("NE",11.25,33.75),
        ("E",33.75,56.25),
        ("SE",56.25,78.75),
        ("S",78.75,101.25),
        ("SW",101.25,123.75),
        ("W",123.75,146.25),
        ("NW",146.25,168.75),
        ("N",168.75,360)    ,
        ("SSE",101.25,123.75),
        ("SSW",123.75,146.25),
        ("NNW",146.25,168.75),
        ("NNE",11.25,33.75),
        ("ENE",33.75,56.25),
        ("ESE",56.25,78.75),
        ("WSW",78.75,101.25),
        ("WNW",101.25,123.75),
        ("SW",123.75,146.25),
        ("NW",146.25,168.75),
        ("NNW",168.75,360   )
    ]
    compass_direction = next(point for point ,start,end in compass_points if start <= wind_deg < end) # determine the compass direction based on the wind gust direction
    compass_direction_encoded = le.transform([compass_direction])[0] if compass_direction in le.classes_ else -1 # encode the compass direction using the label encoder
    # Prepare data for regression model
    current_data = {
        'MinTemp': current_weather['temp_min'],
        'MaxTemp': current_weather['temp_max'],
        'Humidity': current_weather['humidity'],
        'WindGustDir': compass_direction_encoded,
        'WinGustSpeed': current_weather['Wind_Gust_Speed'],
        'Pressure': current_weather['Pressure'],
        'humidity': current_weather['humidity'],
        'Temp': current_weather['current_temp']
    }